# Import Libraries

In [1]:
from src.utils import  load_data, split_input_output, split_train_test, serialize_data, deserialize_data
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd
import yaml
from copy import deepcopy

In [2]:
with open('../config/path.yaml', 'r') as f:
    path = yaml.safe_load(f)

X_train_path = path['data_path']['input']['X_train']
x_test_path = path['data_path']['input']['x_test']
x_valid_path = path['data_path']['input']['x_valid']
y_train_path = path['data_path']['output']['y_train']
y_test_path = path['data_path']['output']['y_test']
y_valid_path = path['data_path']['output']['y_valid']
fname = path['data_path']['data']['raw']['credit_risk']
X_train_prep_path = path['data_path']['data']['processed']['X_train_prep']
X_test_prep_path = path['data_path']['data']['processed']['X_test_prep']
X_valid_prep_path = path['data_path']['data']['processed']['X_valid_prep']
y_train_prep_path = path['data_path']['data']['processed']['y_train_prep']

ohe_home_ownership_path = path['data_path']['models']['ohe_home_ownership']
ohe_loan_intent_path = path['data_path']['models']['ohe_loan_intent']
ohe_loan_grade_path = path['data_path']['models']['ohe_loan_grade']
ohe_default_on_file_path = path['data_path']['models']['ohe_default_on_file']

In [3]:
FNAME = fname
TARGET_COL = "loan_status"
X_TRAIN_PATH = X_train_path
X_TEST_PATH = x_test_path
X_VALID_PATH = x_valid_path
Y_TRAIN_PATH = y_train_path
Y_TEST_PATH = y_test_path
Y_VALID_PATH = y_valid_path
X_TRAIN_PREP_PATH = X_train_prep_path
X_TEST_PREP_PATH = X_test_prep_path
X_VALID_PREP_PATH = X_valid_prep_path
Y_TRAIN_PREP_PATH = y_train_prep_path

OHE_HOME_OWNERSHIP_PATH = ohe_home_ownership_path
OHE_LOAN_INTENT_PATH = ohe_loan_intent_path
OHE_LOAN_GRADE_PATH = ohe_loan_grade_path
OHE_DEFAULT_FILE_ON_PATH = ohe_default_on_file_path

num_col = ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length']
cat_col = ['person_home_ownership', 'loan_grade','cb_person_default_on_file']

In [4]:
X_train = deserialize_data(X_TRAIN_PATH)
X_test = deserialize_data(X_TEST_PATH)
X_valid = deserialize_data(X_VALID_PATH)
y_train = deserialize_data(Y_TRAIN_PATH)
y_test = deserialize_data(Y_TEST_PATH)
y_valid = deserialize_data(Y_VALID_PATH)

Succeed to deserialize data
Succeed to deserialize data
Succeed to deserialize data
Succeed to deserialize data
Succeed to deserialize data
Succeed to deserialize data


In [5]:
type(y_train)

pandas.core.series.Series

In [6]:
y_train.reset_index()

,index,loan_status
0,15884,0
1,15138,1
2,7474,0
3,18212,1
4,6493,0
...,...,...
26059,14621,1
26060,18736,0
26061,1663,0
26062,18257,0


# Function

In [7]:
def drop_duplicate_data(X: pd.DataFrame, y: pd.Series):
    """ 
    To drop duplicated datas

    Parameters:
        X : Input data frame (type: pandas DataFrame)
        y : Output / target data frame (type: pandas DataFrame)

    Return Value:
        X : Input data frame with no duplicated datas (type: pandas DataFrame)
        y : Output / target data frame with no duplicated datas (type: pandas DataFrame)
    """
    
    if type(X) != pd.DataFrame:
        raise RuntimeError("X type not valid!")
    elif type(y) != pd.Series:
        raise RuntimeError("y type not valid!")
    else :
        print("Fungsi drop_duplicate_data: parameter telah divalidasi")

    X = X.copy()
    y = y.copy()
    print(f"Fungsi drop_duplicate_data: shape dataset sebelum dropping duplicate X adalah : {X.shape}")
    X_duplicate = X[X.duplicated(keep= False)]
    print(f"Fungsi drop_duplicate_data: shape dari data X yang duplicate adalah: {X_duplicate.shape}")
    X_clean = tuple()
    X_clean = len(X) - len(X_duplicate)
    print(f"Fungsi drop_duplicate_data: shape dataset X setelah drop duplicate seharusnya adalah: {X_clean}")
    X.drop_duplicates(inplace= True, keep= False)
    print(f"Fungsi drop_duplicate_data: shape dataset X setelah dropping duplicate adalah: {X.shape}")
    y = y.loc[X.index]
    print(f"Fungsi drop_duplicate_data: shape dataset y setelah dropping duplicate adalah: {y.shape}")
    
    return X, y

def median_imputation(data: pd.DataFrame, subset_data, fit: bool):
    """ 
    To calculate or do imputation with median

    Parameters:
        - data : input data (type: pandas DataFrame)
        - subset_data : it depends on fit, if fit = True then it contains columns name with nan value, and if fit = False then it will impute based on subset_data
            (
                type :
                    if fit = True, list
                    if fit = False, dict
            )

    Return Value:
        - if fit = True:
            imputation_data : contains median for each selected column (type: float64)
        - if fit = False:
            data : data with imputed nan column (type: pandas DataFrame)
    """

    if not isinstance(data, pd.DataFrame):
        raise RuntimeError('Fungsi median_imputation: parameter data haruslah bertipe DataFrame!')
    
    if fit == True:
        if not isinstance(subset_data, list):
            raise RuntimeError('Fungsi median_imputation: untuk nilai parameter fit = True, ' \
            'subset_data harus bertipe list dan berisi daftar nama kolom yang ingin dicari nilai mediannya guna menjadi data imputasi pada kolom tersebut.')
        
        print('')
        print("Fungsi median_imputation: parameter telah divalidasi.")
        data = data.copy()
        subset_data = deepcopy(subset_data) # sepertinya menggunakan copy(deep= True) juga bisa bekerja
        imputation_data = dict()

        for subset in subset_data:
            med = data[subset].median()
            imputation_data[subset] = med

        print(f"Fungsi median_imputation: proses fitting telah selesai, berikut hasilnya {imputation_data}")

        return imputation_data
            

    elif fit == False:
        if not isinstance(subset_data, dict):
            raise RuntimeError('Fungsi median_imputation: untuk nilai parameter fit = False, ' \
            'subset_data harus bertipe dict dan berisi key yang merupakan nama kolom beserta value yang merupakan nilai median dari kolom tersebut.')
        
        print('')
        print("Fungsi median_imputation: parameter telah divalidasi.")
        data = data.copy()
        subset_data = deepcopy(subset_data) # sepertinya menggunakan copy(deep= True) juga bisa bekerja

        for subset in subset_data:
            print('Fungsi median_imputation: informasi count na sebelum dilakukan imputasi:')
            print(f"{data[subset].isna().sum()}")
            print('')

        data.fillna(subset_data, inplace= True)
        
        for subset in subset_data:
            print('Fungsi median_imputation: informasi count na setelah dilakukan imputasi:')
            print(f"{data[subset].isna().sum()}")
            print('')

        return data

    else:
        raise RuntimeError('Fungsi median_imputation: parameter fit haruslah bertipe boolean, bernilai True atau False.')
    
def create_onehot_encoder(categories: list, path: str):
    """ 
    Onehot Encoder for categorical data

    Parameters:
        - categories : categorical data (type: list)
        - path : path to saved ohe (type: str)

    Return Value:
        - ohe : onehot encoder (type: sklearn OneHotEncoder)
    """
    
    if not isinstance(categories, list):
        raise RuntimeError('Fungsi create_onehot_encoder: parameter categories haruslah bertipe list, berisi kategori yang akan dibuat encodernya.')
    elif any(isinstance(item, list) for item in categories):
        raise ValueError('Fungsi create_onehot_encoder: parameter categories memiliki dimensi lebih dari 1!')

    if not isinstance(path, str):
        raise RuntimeError('Fungsi create_onehot_encoder: parameter path haruslah bertipe string, berisi lokasi pada disk komputer dimana encoder akan disimpan.')
    
    ohe = OneHotEncoder()
    categories = np.array(categories).reshape(-1, 1)
    ohe.fit(categories)
    serialize_data(ohe, path)
    print(f"Kategori yang telah dipelajari adalah {ohe.categories_[0].tolist()}")

    return ohe

def ohe_transform(dataset: pd.DataFrame, subset: str, prefix: str, ohe: OneHotEncoder):
    """  
    Transforing OHE into dataset

    Parameters :
        - dataset : input data (type: pandas Dataframe)
        - subset : selected column to implement transformation (type: string)
        - prefix : prefix for OHE transformation (type: string)
        - ohe : selected OneHotEncoder (type: sklearn OneHotEncoder)

    Return Value:
        - dataset : transformed data with OHE (type: pandas DataFrame)
    """

    if not isinstance(dataset, pd.DataFrame):
        raise RuntimeError('Fungsi ohe_transform: parameter dataset harus bertipe DataFrame!')
    
    if not isinstance(ohe, OneHotEncoder):
        raise RuntimeError('Fungsi ohe_transform: parameter ohe harus bertipe OneHotEncoder!')
    
    if not isinstance(prefix, str):
        raise RuntimeError('Fungsi ohe_transform: parameter prefix harus bertipe str!')
    
    if not isinstance(subset, str):
        raise RuntimeError('Fungsi ohe_transform: parameter subset harus bertipe str!')
    
    try:
        column_list = dataset.columns.to_list()
        column_list.index(subset)
    except:
        raise RuntimeError('Fungsi ohe_transform: parameter subset string namun data tidak ditemukan dalam daftar kolom yang terdapat pada parameter dataset.')
    
    print('Fungsi ohe_transform: parameter telah divalidasi.')
    dataset = dataset.copy()
    print(f"Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah {dataset.columns.to_list()}")
    col_names = [prefix + '_' + col_name for col_name in ohe.categories_[0].tolist()]
    
    encoded = pd.DataFrame(ohe.transform(dataset[[subset]]).toarray(), columns= col_names, index= dataset.index)
    dataset = pd.concat([dataset, encoded], axis= 1)
    dataset.drop(columns= [subset], inplace= True)
    print(f"Fungsi ohe_transform: daftar nama kolom setelah dilakukan pengkodean adalah {dataset.columns}")
    return dataset


In [8]:
X_train, y_train = drop_duplicate_data(X_train, y_train)

Fungsi drop_duplicate_data: parameter telah divalidasi
Fungsi drop_duplicate_data: shape dataset sebelum dropping duplicate X adalah : (26064, 11)
Fungsi drop_duplicate_data: shape dari data X yang duplicate adalah: (192, 11)
Fungsi drop_duplicate_data: shape dataset X setelah drop duplicate seharusnya adalah: 25872
Fungsi drop_duplicate_data: shape dataset X setelah dropping duplicate adalah: (25872, 11)
Fungsi drop_duplicate_data: shape dataset y setelah dropping duplicate adalah: (25872,)


In [9]:
subset_data = X_train.columns[X_train.isna().any()].to_list()
subset_data = median_imputation(X_train, subset_data, True)
X_train = median_imputation(X_train, subset_data, False)


Fungsi median_imputation: parameter telah divalidasi.
Fungsi median_imputation: proses fitting telah selesai, berikut hasilnya {'person_emp_length': np.float64(4.0), 'loan_int_rate': np.float64(10.99)}

Fungsi median_imputation: parameter telah divalidasi.
Fungsi median_imputation: informasi count na sebelum dilakukan imputasi:
730

Fungsi median_imputation: informasi count na sebelum dilakukan imputasi:
2481

Fungsi median_imputation: informasi count na setelah dilakukan imputasi:
0

Fungsi median_imputation: informasi count na setelah dilakukan imputasi:
0



In [10]:
X_test = median_imputation(X_test, subset_data, False)


Fungsi median_imputation: parameter telah divalidasi.
Fungsi median_imputation: informasi count na sebelum dilakukan imputasi:
77

Fungsi median_imputation: informasi count na sebelum dilakukan imputasi:
303

Fungsi median_imputation: informasi count na setelah dilakukan imputasi:
0

Fungsi median_imputation: informasi count na setelah dilakukan imputasi:
0



In [11]:
X_valid = median_imputation(X_valid, subset_data, False)


Fungsi median_imputation: parameter telah divalidasi.
Fungsi median_imputation: informasi count na sebelum dilakukan imputasi:
80

Fungsi median_imputation: informasi count na sebelum dilakukan imputasi:
312

Fungsi median_imputation: informasi count na setelah dilakukan imputasi:
0

Fungsi median_imputation: informasi count na setelah dilakukan imputasi:
0



In [12]:
person_home_ownership = X_train['person_home_ownership'].to_list()
loan_intent = X_train['loan_intent'].to_list()
loan_grade = X_train['loan_grade'].to_list()
cb_person_default_on_file = X_train['cb_person_default_on_file'].to_list()

ohe_home_ownership = create_onehot_encoder(person_home_ownership, OHE_HOME_OWNERSHIP_PATH)
ohe_loan_intent = create_onehot_encoder(loan_intent, OHE_LOAN_INTENT_PATH)
ohe_loan_grade = create_onehot_encoder(loan_grade, OHE_LOAN_GRADE_PATH)
ohe_default_on_file = create_onehot_encoder(cb_person_default_on_file, OHE_DEFAULT_FILE_ON_PATH)

Succeed to serialize data
Kategori yang telah dipelajari adalah ['MORTGAGE', 'OTHER', 'OWN', 'RENT']
Succeed to serialize data
Kategori yang telah dipelajari adalah ['DEBTCONSOLIDATION', 'EDUCATION', 'HOMEIMPROVEMENT', 'MEDICAL', 'PERSONAL', 'VENTURE']
Succeed to serialize data
Kategori yang telah dipelajari adalah ['A', 'B', 'C', 'D', 'E', 'F', 'G']
Succeed to serialize data
Kategori yang telah dipelajari adalah ['N', 'Y']


In [13]:
print(type(ohe_home_ownership))

<class 'sklearn.preprocessing._encoders.OneHotEncoder'>


In [14]:
X_train.columns

Index(['person_age', 'person_income', 'person_home_ownership',
       'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length'],
      dtype='object')

In [15]:
X_train = ohe_transform(X_train, 'person_home_ownership', 'home_ownership', ohe_home_ownership)

Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']
Fungsi ohe_transform: daftar nama kolom setelah dilakukan pengkodean adalah Index(['person_age', 'person_income', 'person_emp_length', 'loan_intent',
       'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length',
       'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT'],
      dtype='object')


/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [16]:
X_train = ohe_transform(X_train, 'loan_intent', 'loan_intent', ohe_loan_intent)

Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT']
Fungsi ohe_transform: daftar nama kolom setelah dilakukan pengkodean adalah Index(['person_age', 'person_income', 'person_emp_length', 'loan_grade',
       'loan_amnt', 'loan_int_rate', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length',
       'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION',
       'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT',
       'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE'],
      dtype='object')


/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [17]:
X_train = ohe_transform(X_train, 'loan_grade', 'loan_grade', ohe_loan_grade)

Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_emp_length', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']
Fungsi ohe_transform: daftar nama kolom setelah dilakukan pengkodean adalah Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file',
       'cb_person_cred_hist_length', 'home_ownership_MORTGAGE',
       'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT',
       'loan_intent_DEBTCONSOLIDATION', 'loan_intent_EDUCATION',
       'loa

/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [18]:
X_train = ohe_transform(X_train, 'cb_person_default_on_file', 'default_onfile', ohe_default_on_file)

Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN', 'home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE', 'loan_grade_A', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D', 'loan_grade_E', 'loan_grade_F', 'loan_grade_G']
Fungsi ohe_transform: daftar nama kolom setelah dilakukan pengkodean adalah Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT', 'loan_i

/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [19]:
X_train.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION',
       'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT',
       'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE',
       'loan_grade_A', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D',
       'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'default_onfile_N',
       'default_onfile_Y'],
      dtype='object')

In [20]:
X_test = ohe_transform(X_test, 'person_home_ownership', 'home_ownership', ohe_home_ownership)
X_test = ohe_transform(X_test, 'loan_intent', 'loan_intent', ohe_loan_intent)
X_test = ohe_transform(X_test, 'loan_grade', 'loan_grade', ohe_loan_grade)
X_test = ohe_transform(X_test, 'cb_person_default_on_file', 'default_onfile', ohe_default_on_file)

Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']
Fungsi ohe_transform: daftar nama kolom setelah dilakukan pengkodean adalah Index(['person_age', 'person_income', 'person_emp_length', 'loan_intent',
       'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length',
       'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT'],
      dtype='object')
Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_r

/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [21]:
X_test.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION',
       'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT',
       'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE',
       'loan_grade_A', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D',
       'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'default_onfile_N',
       'default_onfile_Y'],
      dtype='object')

In [22]:
X_valid = ohe_transform(X_valid, 'person_home_ownership', 'home_ownership', ohe_home_ownership)
X_valid = ohe_transform(X_valid, 'loan_intent', 'loan_intent', ohe_loan_intent)
X_valid = ohe_transform(X_valid, 'loan_grade', 'loan_grade', ohe_loan_grade)
X_valid = ohe_transform(X_valid, 'cb_person_default_on_file', 'default_onfile', ohe_default_on_file)

Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length']
Fungsi ohe_transform: daftar nama kolom setelah dilakukan pengkodean adalah Index(['person_age', 'person_income', 'person_emp_length', 'loan_intent',
       'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income',
       'cb_person_default_on_file', 'cb_person_cred_hist_length',
       'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT'],
      dtype='object')
Fungsi ohe_transform: parameter telah divalidasi.
Fungsi ohe_transform: daftar nama kolom sebelum dilakukan pengkodean adalah ['person_age', 'person_income', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_r

/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(
/home/vega33/VEGA_ML_PROCESS/vega_venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2742: UserWarning: X has feature names, but OneHotEncoder was fitted without feature names
  warnings.warn(


In [23]:
X_valid.columns

Index(['person_age', 'person_income', 'person_emp_length', 'loan_amnt',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'home_ownership_MORTGAGE', 'home_ownership_OTHER', 'home_ownership_OWN',
       'home_ownership_RENT', 'loan_intent_DEBTCONSOLIDATION',
       'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT',
       'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE',
       'loan_grade_A', 'loan_grade_B', 'loan_grade_C', 'loan_grade_D',
       'loan_grade_E', 'loan_grade_F', 'loan_grade_G', 'default_onfile_N',
       'default_onfile_Y'],
      dtype='object')

In [24]:
serialize_data(X_train, X_TRAIN_PREP_PATH)
serialize_data(X_test, X_TEST_PREP_PATH)
serialize_data(X_valid, X_VALID_PREP_PATH)

Succeed to serialize data
Succeed to serialize data
Succeed to serialize data


In [25]:
serialize_data(y_train, Y_TRAIN_PREP_PATH)

Succeed to serialize data
